In [ ]:
from pyscf import gto, scf

from qarp.blocks import UCCBlock, MappedONVStateBlock, CompositeBlock
from qarp.operators import JordanWigner
from qarp.operators.pyscf import fermion_operator_from_mf, onv_from_mf
from qarp.algorithms import VQD


bl = 0.735
geometry = f"H 0 0 0; H 0 0 {bl}"
mol = gto.M(atom=geometry, basis="sto3g", verbose=-1, symmetry=True)
mol.build()
mf = scf.RHF(mol)
mf.kernel()

onv = onv_from_mf(mf)
ucc = UCCBlock(onv, generalised=True)
ref = MappedONVStateBlock(onv)
wfn = CompositeBlock([ref, ucc]).build()

fermion_operator = fermion_operator_from_mf(mf)
qop = JordanWigner().encode_operator(fermion_operator)


In [ ]:
kets = [wfn.refresh_symbols(f"_{i}").build() for i in range(4)]
vqd = VQD(qop, kets=kets, weights=[3, 3, 3], gradient=False, verbose=True)
vqd.build()
e_vqd, x_vqd = vqd.run()